# Pipeline Data Mining: Prediksi Klaim Asuransi Kesehatan
## Kompetisi MCF ITB — Forecasting Claim Frequency, Severity & Total Claim

**Abstrak** — Notebook ini menyajikan pipeline end-to-end untuk memprediksi tiga metrik klaim asuransi kesehatan bulanan (Agustus–Desember 2025): *Claim Frequency*, *Claim Severity*, dan *Total Claim*. Pendekatan yang digunakan mencakup eksplorasi data, *feature engineering* berbasis domain aktuaria, pemodelan ensemble (*LightGBM*, *XGBoost*, *CatBoost*) yang dikombinasikan dengan metode statistik (*Exponential Smoothing*), serta *hyperparameter tuning* menggunakan Optuna. Analisis kepentingan fitur dilakukan dengan SHAP values untuk interpretabilitas model.

---

## 1. Instalasi & Import Library

**Rasionalisasi:** Langkah pertama adalah memastikan seluruh dependensi tersedia. Kita menggunakan:
- **pandas/numpy** untuk manipulasi data tabular.
- **matplotlib/seaborn** untuk visualisasi eksploratif.
- **scikit-learn** untuk utilitas validasi, metrik evaluasi, dan **Ridge Regression** (model linear regularized).
- **scipy.optimize** untuk **optimasi bobot ensemble** secara data-driven (MAPE minimization).
- **LightGBM, XGBoost, CatBoost** — tiga algoritma gradient boosting *state-of-the-art* untuk data tabular.
- **Optuna** untuk *Bayesian hyperparameter optimization* yang lebih efisien dibanding grid/random search.
- **SHAP** untuk interpretasi model berbasis teori kooperatif (*Shapley values*).
- **statsmodels** untuk metode statistik *Exponential Smoothing* sebagai baseline forecasting.

In [ ]:
# Instalasi library yang mungkin belum tersedia
!pip install lightgbm xgboost catboost optuna shap statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 25.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap
from statsmodels.tsa.holtwinters import ExponentialSmoothing

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Semua library berhasil dimuat.')


Semua library berhasil dimuat.


In [ ]:
drive.mount('/content/drive')


## 2. Identifikasi Target Prediksi

**Rasionalisasi:** Sebelum melakukan analisis data, kita **wajib** memahami apa yang harus diprediksi. Dengan membaca `sample_submission.csv`, kita dapat mengidentifikasi:
1. Format output yang diharapkan.
2. Variabel target (*Claim Frequency*, *Claim Severity*, *Total Claim*).
3. Horizon prediksi (bulan apa saja yang harus di-forecast).
4. Tipe masalah (regresi time-series).

Informasi ini menentukan seluruh strategi pemodelan yang akan digunakan.

In [ ]:
df_sub = pd.read_csv('/content/drive/MyDrive/MCF ITB/sample_submission.csv')

print('=' * 65)
print('ANALISIS FORMAT SUBMISSION')
print('=' * 65)
print(f'Dimensi : {df_sub.shape}')
print(f'Kolom   : {df_sub.columns.tolist()}')
print()
display(df_sub)

# Parsing komponen ID
df_sub['year']   = df_sub['id'].apply(lambda x: int(x.split('_')[0]))
df_sub['month']  = df_sub['id'].apply(lambda x: int(x.split('_')[1]))
df_sub['metric'] = df_sub['id'].apply(lambda x: '_'.join(x.split('_')[2:]))

print(f"\nMetrik yang diprediksi : {df_sub['metric'].unique().tolist()}")
print(f"Bulan prediksi         : {sorted(df_sub['month'].unique().tolist())}")
print(f"Tahun                  : {df_sub['year'].unique().tolist()}")

print('\n>>> KESIMPULAN:')
print('    Tipe Masalah    : TIME SERIES FORECASTING (Regresi)')
print('    Target          : Claim_Frequency, Claim_Severity, Total_Claim (agregat bulanan)')
print('    Horizon         : Agustus - Desember 2025 (5 bulan ke depan)')
print('    Metrik Evaluasi : RMSE / MAE (regresi) — kita optimasi RMSE')

## 3. Pemuatan Data & Exploratory Data Analysis (EDA)

**Rasionalisasi:** EDA merupakan tahap fundamental dalam setiap proyek data science (*Tukey, 1977*). Tujuannya:
1. Memahami distribusi, tipe, dan skala setiap variabel.
2. Mengidentifikasi *missing values*, *outlier*, dan anomali.
3. Menemukan pola awal yang dapat menginformasikan strategi *feature engineering*.
4. Memvalidasi asumsi domain (misal: relasi antara Data Klaim dan Data Polis).

In [ ]:
# ── Muat Data ──────────────────────────────────────────────────────
df_klaim = pd.read_csv('/content/drive/MyDrive/MCF ITB/Data_Klaim.csv')
df_polis = pd.read_csv('/content/drive/MyDrive/MCF ITB/Data_Polis.csv')

print('=' * 65)
print('DATA KLAIM')
print('=' * 65)
print(f'Dimensi : {df_klaim.shape}')
print(f'\nTipe data:\n{df_klaim.dtypes}')
print(f'\nMissing values:\n{df_klaim.isnull().sum()}')
display(df_klaim.head())

print('\n' + '=' * 65)
print('DATA POLIS')
print('=' * 65)
print(f'Dimensi : {df_polis.shape}')
print(f'\nTipe data:\n{df_polis.dtypes}')
print(f'\nMissing values:\n{df_polis.isnull().sum()}')
display(df_polis.head())

# ═══════════════════════════════════════════════════════════════════
# DATA QUALITY AUDIT
# ═══════════════════════════════════════════════════════════════════
print('\n' + '=' * 65)
print('DATA QUALITY AUDIT — DATA KLAIM')
print('=' * 65)

# 1. Status Klaim
print(f'\n[1] Status Klaim:')
print(df_klaim['Status Klaim'].value_counts())

# 2. Inpatient/Outpatient unique values
print(f'\n[2] Inpatient/Outpatient — Unique values:')
print(df_klaim['Inpatient/Outpatient'].fillna('(KOSONG)').value_counts())

# 3. Lokasi RS
print(f'\n[3] Lokasi RS — Unique values:')
print(df_klaim['Lokasi RS'].fillna('(KOSONG)').value_counts())

# 4. Nominal anomali
zero_biaya = (df_klaim['Nominal Biaya RS Yang Terjadi'] == 0).sum()
zero_klaim = (df_klaim['Nominal Klaim Yang Disetujui'] == 0).sum()
print(f'\n[4] Anomali Finansial:')
print(f'  Nominal Biaya RS = 0     : {zero_biaya} baris')
print(f'  Nominal Klaim Disetujui = 0 : {zero_klaim} baris')

# 5. Rasio klaim ekstrem
_biaya_nonzero = df_klaim['Nominal Biaya RS Yang Terjadi'].replace(0, np.nan)
_rasio = df_klaim['Nominal Klaim Yang Disetujui'] / _biaya_nonzero
print(f'\n[5] Rasio Klaim (Disetujui / Biaya RS):')
print(f'  Min    : {_rasio.min():.4f}')
print(f'  Median : {_rasio.median():.4f}')
print(f'  Max    : {_rasio.max():.2f}')
print(f'  Rasio > 5 : {(_rasio > 5).sum()} baris')

# 6. Polis match check
klaim_polis_set = set(df_klaim['Nomor Polis'].unique())
polis_polis_set = set(df_polis['Nomor Polis'].unique())
only_in_klaim = klaim_polis_set - polis_polis_set
print(f'\n[6] Konsistensi Polis:')
print(f'  Polis unik di klaim              : {len(klaim_polis_set)}')
print(f'  Polis unik di polis              : {len(polis_polis_set)}')
print(f'  Di klaim tapi TIDAK di polis     : {len(only_in_klaim)}')
print(f'  Di polis tapi TIDAK punya klaim  : {len(polis_polis_set - klaim_polis_set)}')

# 7. Klaim per polis
klaim_freq = df_klaim['Nomor Polis'].value_counts()
print(f'\n[7] Distribusi Klaim per Polis:')
print(f'  Min/Median/Max : {klaim_freq.min()} / {klaim_freq.median():.0f} / {klaim_freq.max()}')
print(f'  Top 5 heavy claimers:')
for p, c in klaim_freq.head(5).items():
    print(f'    {p}: {c} klaim')

# 8. Tanggal check
df_klaim['_tgl_masuk'] = pd.to_datetime(df_klaim['Tanggal Pasien Masuk RS'], errors='coerce')
df_klaim['_tgl_keluar'] = pd.to_datetime(df_klaim['Tanggal Pasien Keluar RS'], errors='coerce')
ip_zero_stay = ((df_klaim['Inpatient/Outpatient'] == 'IP') &
                (df_klaim['_tgl_keluar'] == df_klaim['_tgl_masuk'])).sum()
print(f'\n[8] Anomali Tanggal:')
print(f'  Tanggal Keluar < Tanggal Masuk   : {(df_klaim["_tgl_keluar"] < df_klaim["_tgl_masuk"]).sum()}')
print(f'  Inpatient dengan 0 hari rawat    : {ip_zero_stay}')
print(f'  Range Tanggal Masuk RS           : {df_klaim["_tgl_masuk"].min().date()} — {df_klaim["_tgl_masuk"].max().date()}')

df_klaim.drop(columns=['_tgl_masuk', '_tgl_keluar'], inplace=True)

print('\n' + '=' * 65)
print('DATA QUALITY AUDIT — DATA POLIS')
print('=' * 65)
print(f'Missing values : {df_polis.isnull().sum().sum()} (TIDAK ADA)')
print(f'\nPlan Code   : {df_polis["Plan Code"].value_counts().to_dict()}')
print(f'Gender      : {df_polis["Gender"].value_counts().to_dict()}')
print(f'Domisili    : {df_polis["Domisili"].nunique()} kota unik')

In [ ]:
# ── Distribusi Variabel Kategorikal ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

cat_cols_klaim = ['Reimburse/Cashless', 'Inpatient/Outpatient', 'Lokasi RS']
cat_cols_polis = ['Plan Code', 'Gender']

for i, col in enumerate(cat_cols_klaim):
    ax = axes[0][i]
    df_klaim[col].fillna('(KOSONG)').value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('husl', 8))
    ax.set_title(f'Distribusi {col}', fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

for j, col in enumerate(cat_cols_polis):
    ax = axes[1][j]
    df_polis[col].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('husl', 8))
    ax.set_title(f'Distribusi {col}', fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

# Klaim per bulan (foreshadow time series)
ax = axes[1][2]
ym_counts = pd.to_datetime(df_klaim['Tanggal Pasien Masuk RS']).dt.to_period('M').value_counts().sort_index()
ym_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Klaim per Bulan', fontweight='bold')
ax.tick_params(axis='x', rotation=45)

plt.suptitle('Distribusi Variabel Kategorikal & Temporal', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Distribusi Variabel Numerik ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_klaim['Nominal Klaim Yang Disetujui'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Nominal Klaim Disetujui', fontweight='bold')
axes[0].set_xlabel('Nominal (Rp)')

df_klaim['Nominal Biaya RS Yang Terjadi'].hist(bins=50, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Distribusi Nominal Biaya RS', fontweight='bold')
axes[1].set_xlabel('Nominal (Rp)')
plt.tight_layout()
plt.show()

## 4. Data Merging & Comprehensive Data Cleaning

**Rasionalisasi:** Berdasarkan audit kualitas data pada tahap sebelumnya, kami mengidentifikasi **7 isu kritis** yang memerlukan penanganan strategis. Setiap keputusan diambil berdasarkan prinsip **memaksimalkan akurasi prediksi agregat bulanan** tanpa kehilangan informasi penting.

### 4a. Merge Strategy
Penggabungan Data Klaim dan Data Polis melalui kunci `Nomor Polis` menggunakan **left join** — mempertahankan seluruh 4.627 klaim karena setiap baris berkontribusi pada *Claim_Frequency* bulanan. Audit menunjukkan 0 polis klaim yang tidak cocok (semua ada di Data Polis).

### 4b. Strategi Cleaning — Keputusan per Isu

| # | Isu | Jumlah | Keputusan | Alasan |
|---|-----|--------|-----------|--------|
| 1 | **37 baris kosong `Inpatient/Outpatient`** | 37 (0.8%) | Imputasi berdasarkan Lama Rawat: jika masuk=keluar → `OP`, jika >0 hari → `IP` | Mempertahankan baris untuk frekuensi; pola rawat menentukan kategori klinis. |
| 2 | **37 baris kosong `Tanggal Pembayaran`** | 37 (0.8%) | Imputasi dengan **median delay** dari Tgl Keluar RS | Metode robust terhadap outlier; baris yang sama dengan isu #1. |
| 3 | **6 baris kosong `ICD Diagnosis`** | 6 (0.1%) | Isi dengan `'Z99'` (kategori *unspecified*) | Minimal impact pada agregasi; mempertahankan baris. |
| 4 | **7 baris kosong `Lokasi RS`** | 7 (0.2%) | Imputasi berdasarkan **Plan Code**: M-003→Indonesia, lainnya→mode per plan | Plan Code menentukan cakupan geografis; korelasi kuat dengan lokasi. |
| 5 | **Lokasi RS terfragmentasi** (11 kategori) | — | Konsolidasi → **4 kategori**: Indonesia, Singapore, Malaysia, Overseas | Kategori kecil (Taiwan, HK, Japan, dll) total hanya 15 baris; *high cardinality* menambah noise. |
| 6 | **Nominal Biaya RS = 0** | 2 baris | Ganti 0 → NaN untuk rasio saja; **pertahankan baris** | Mencegah division-by-zero pada rasio; baris tetap berkontribusi ke frekuensi & total. |
| 7 | **Rasio klaim ekstrem** (>5×) | 3 baris | **Winsorize** rasio pada persentil 1–99 | Menghilangkan pengaruh outlier finansial pada fitur agregat tanpa menghapus data. |

### 4c. Keputusan: TIDAK di-drop / TIDAK diubah
- **13 klaim dengan Nominal Disetujui = 0**: Ini adalah klaim yang sah tetapi ditolak — tetap dihitung dalam frekuensi.
- **762 IP dengan 0 hari rawat**: Umum di Indonesia untuk prosedur singkat (*day-case surgery*) — biarkan apa adanya.
- **Status Klaim**: 100% = `PAID` → **drop kolom** (*zero variance*, tidak informatif).
- **Heavy claimers** (POL-2078: 222 klaim): Mencerminkan realitas pasien kronis (dialisis/kemoterapi) — **pertahankan** karena mempengaruhi frekuensi bulanan secara nyata.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 4a. MERGE DATA KLAIM + DATA POLIS
# ═══════════════════════════════════════════════════════════════════
df = df_klaim.merge(df_polis, on='Nomor Polis', how='left')
print(f'Dimensi setelah merge : {df.shape}')
print(f'Polis unik di klaim   : {df_klaim["Nomor Polis"].nunique()}')
print(f'Polis unik di polis   : {df_polis["Nomor Polis"].nunique()}')
print(f'Unmatched polis       : {df["Plan Code"].isna().sum()} (expected: 0)')
print(f'\nMissing SEBELUM cleaning:\n{df.isnull().sum()}\n')

# ═══════════════════════════════════════════════════════════════════
# STEP 4b. PARSING TANGGAL
# ═══════════════════════════════════════════════════════════════════
date_cols_klaim = ['Tanggal Pembayaran Klaim', 'Tanggal Pasien Masuk RS', 'Tanggal Pasien Keluar RS']
for col in date_cols_klaim:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df['Tanggal Lahir'] = pd.to_datetime(df['Tanggal Lahir'].astype(str), format='%Y%m%d', errors='coerce')
df['Tanggal Efektif Polis'] = pd.to_datetime(df['Tanggal Efektif Polis'].astype(str), format='%Y%m%d', errors='coerce')

# Parsing juga di df_polis untuk pemakaian fitur polis bulanan nanti
df_polis['Tanggal Lahir'] = pd.to_datetime(df_polis['Tanggal Lahir'].astype(str), format='%Y%m%d', errors='coerce')
df_polis['Tanggal Efektif Polis'] = pd.to_datetime(df_polis['Tanggal Efektif Polis'].astype(str), format='%Y%m%d', errors='coerce')

print('Parsing tanggal selesai.')

# ═══════════════════════════════════════════════════════════════════
# STEP 4c. CLEANING ISU #1 — IMPUTASI INPATIENT/OUTPATIENT (37 baris)
# ═══════════════════════════════════════════════════════════════════
missing_io = df['Inpatient/Outpatient'].isna() | (df['Inpatient/Outpatient'].str.strip() == '')
lama_rawat_temp = (df['Tanggal Pasien Keluar RS'] - df['Tanggal Pasien Masuk RS']).dt.days

# Logika: jika rawat > 0 hari → IP (Inpatient), jika = 0 → OP (Outpatient)
df.loc[missing_io & (lama_rawat_temp > 0), 'Inpatient/Outpatient'] = 'IP'
df.loc[missing_io & (lama_rawat_temp == 0), 'Inpatient/Outpatient'] = 'OP'
# Fallback jika lama rawat tidak tersedia
df.loc[df['Inpatient/Outpatient'].isna() | (df['Inpatient/Outpatient'].str.strip() == ''), 'Inpatient/Outpatient'] = 'OP'

print(f'[Isu #1] Imputasi Inpatient/Outpatient: {missing_io.sum()} baris → selesai')
print(f'  Distribusi setelah imputasi: {df["Inpatient/Outpatient"].value_counts().to_dict()}')

# ═══════════════════════════════════════════════════════════════════
# STEP 4d. CLEANING ISU #2 — IMPUTASI TANGGAL PEMBAYARAN (37 baris)
# ═══════════════════════════════════════════════════════════════════
paid_mask = df['Tanggal Pembayaran Klaim'].notna() & df['Tanggal Pasien Keluar RS'].notna()
df.loc[paid_mask, '_delay'] = (
    df.loc[paid_mask, 'Tanggal Pembayaran Klaim'] - df.loc[paid_mask, 'Tanggal Pasien Keluar RS']
).dt.days
median_delay = df['_delay'].median()

missing_pay = df['Tanggal Pembayaran Klaim'].isna()
df.loc[missing_pay, 'Tanggal Pembayaran Klaim'] = (
    df.loc[missing_pay, 'Tanggal Pasien Keluar RS'] + pd.Timedelta(days=median_delay)
)
df.drop(columns=['_delay'], inplace=True)

print(f'\n[Isu #2] Imputasi Tanggal Pembayaran Klaim: {missing_pay.sum()} baris')
print(f'  Median payment delay: {median_delay:.0f} hari')

# ═══════════════════════════════════════════════════════════════════
# STEP 4e. CLEANING ISU #3 — IMPUTASI ICD DIAGNOSIS (6 baris)
# ═══════════════════════════════════════════════════════════════════
missing_icd = df['ICD Diagnosis'].isna() | (df['ICD Diagnosis'].str.strip() == '')
df.loc[missing_icd, 'ICD Diagnosis'] = 'Z99'
df.loc[missing_icd, 'ICD Description'] = 'Unspecified'
print(f'\n[Isu #3] Imputasi ICD Diagnosis: {missing_icd.sum()} baris → "Z99"')

# ═══════════════════════════════════════════════════════════════════
# STEP 4f. CLEANING ISU #4 — IMPUTASI LOKASI RS (7 baris)
# ═══════════════════════════════════════════════════════════════════
missing_loc = df['Lokasi RS'].isna() | (df['Lokasi RS'].str.strip() == '')
# Imputasi berdasarkan Plan Code: M-003 = Indonesia-only, M-002 = Asia, M-001 = Worldwide
# Hitung mode Lokasi RS per Plan Code dari data yang ada
loc_by_plan = df[~missing_loc].groupby('Plan Code')['Lokasi RS'].agg(lambda x: x.mode().iloc[0])
print(f'\n[Isu #4] Mode Lokasi RS per Plan Code: {loc_by_plan.to_dict()}')

for idx in df[missing_loc].index:
    plan = df.loc[idx, 'Plan Code']
    df.loc[idx, 'Lokasi RS'] = loc_by_plan.get(plan, 'Indonesia')

print(f'  Imputasi Lokasi RS: {missing_loc.sum()} baris → selesai')

# ═══════════════════════════════════════════════════════════════════
# STEP 4g. CLEANING ISU #5 — KONSOLIDASI LOKASI RS → 4 KATEGORI
# ═══════════════════════════════════════════════════════════════════
print(f'\n[Isu #5] Konsolidasi Lokasi RS:')
print(f'  SEBELUM: {df["Lokasi RS"].nunique()} kategori')

overseas_labels = ['Overseas', 'Others', 'Taiwan', 'Hong Kong', 'Japan', 'Tiongkok', 'Australia']
df['Lokasi RS'] = df['Lokasi RS'].apply(
    lambda x: 'Overseas' if x in overseas_labels else x
)

print(f'  SETELAH: {df["Lokasi RS"].nunique()} kategori → {df["Lokasi RS"].value_counts().to_dict()}')

# ═══════════════════════════════════════════════════════════════════
# STEP 4h. CLEANING ISU #6 — NOMINAL BIAYA RS = 0 → NaN
# ═══════════════════════════════════════════════════════════════════
n_zero_biaya = (df['Nominal Biaya RS Yang Terjadi'] == 0).sum()
df['Nominal Biaya RS Yang Terjadi'] = df['Nominal Biaya RS Yang Terjadi'].replace(0, np.nan)
print(f'\n[Isu #6] Biaya RS = 0 → NaN: {n_zero_biaya} baris')

# ═══════════════════════════════════════════════════════════════════
# STEP 4i. DROP KOLOM TIDAK INFORMATIF
# ═══════════════════════════════════════════════════════════════════
df.drop(columns=['Status Klaim'], inplace=True)
print(f'\nDrop kolom "Status Klaim" (zero variance — 100% PAID)')

# ═══════════════════════════════════════════════════════════════════
# RINGKASAN CLEANING
# ═══════════════════════════════════════════════════════════════════
print(f'\n{"=" * 65}')
print(f'RINGKASAN SETELAH CLEANING')
print(f'{"=" * 65}')
print(f'Dimensi         : {df.shape}')
print(f'Baris hilang    : 0 (TIDAK ada baris yang di-drop)')
print(f'Rentang tanggal : {df["Tanggal Pasien Masuk RS"].min().date()} — {df["Tanggal Pasien Masuk RS"].max().date()}')
print(f'\nMissing values SETELAH cleaning:')
print(df.isnull().sum())

## 5. Feature Engineering — Tingkat Klaim (*Claim-Level*)

**Rasionalisasi:** Fitur-fitur turunan di bawah ini dirancang berdasarkan prinsip aktuaria dan domain asuransi kesehatan. Setiap fitur dipilih karena memiliki **interpretasi bisnis yang jelas** dan **tersedia pada level agregat** untuk bulan-bulan prediksi.

| Fitur | Formula | Alasan |
|-------|---------|--------|
| **Usia Saat Klaim** | (Tgl Masuk RS − Tgl Lahir) / 365.25 | Usia adalah prediktor utama risiko kesehatan (WHO, 2020). |
| **Lama Rawat Inap (LOS)** | Tgl Keluar RS − Tgl Masuk RS | Indikator keparahan penyakit; berkorelasi positif dengan biaya klaim. |
| **Lama Polis Aktif** | (Tgl Masuk RS − Tgl Efektif Polis) / 365.25 | Pemegang polis lama cenderung memiliki pola klaim berbeda (*moral hazard* vs *adverse selection*). |
| **Rasio Klaim** *(winsorized P1–P99)* | Nominal Disetujui / Nominal Biaya RS | Mengukur *claim approval rate*; **rasio di-cap** pada persentil 1–99 untuk menghilangkan outlier ekstrem (rasio >11.000×) yang teridentifikasi di audit. |
| **Selisih Klaim** | Nominal Disetujui − Nominal Biaya RS | Mengukur *cost gap* antara persetujuan dan biaya aktual. |
| **Kategori ICD** | Huruf pertama kode ICD | Mengelompokkan 754 kode ICD ke dalam ~20 kategori besar (misal: C = Neoplasma, A = Infeksi). |
| **Kategori Perawatan** | IP, OP, ODC, ODS | 4 kategori lengkap: *Inpatient*, *Outpatient*, *One Day Care*, *One Day Surgery* — masing-masing memiliki profil biaya berbeda. |
| **Lokasi RS** *(konsolidasi)* | Indonesia / Singapore / Malaysia / Overseas | Menggunakan 4 kategori yang sudah dikonsolidasi. |
| **Binary Flags** | is_inpatient, is_cashless, is_odc_ods | Mempermudah agregasi proporsi bulanan. |

In [ ]:
# ── Feature Engineering Claim-Level ────────────────────────────────

# 1. Usia pasien saat klaim (tahun)
df['Usia_Saat_Klaim'] = (df['Tanggal Pasien Masuk RS'] - df['Tanggal Lahir']).dt.days / 365.25

# 2. Lama Rawat Inap (Length of Stay, hari)
df['Lama_Rawat_Inap'] = (df['Tanggal Pasien Keluar RS'] - df['Tanggal Pasien Masuk RS']).dt.days

# 3. Lama polis aktif saat klaim (tahun)
df['Lama_Polis_Aktif'] = (df['Tanggal Pasien Masuk RS'] - df['Tanggal Efektif Polis']).dt.days / 365.25

# 4. Rasio klaim (Approved / Billed) — ISU #7: WINSORIZE P1-P99
df['Rasio_Klaim'] = df['Nominal Klaim Yang Disetujui'] / df['Nominal Biaya RS Yang Terjadi']
p1  = df['Rasio_Klaim'].quantile(0.01)
p99 = df['Rasio_Klaim'].quantile(0.99)
n_clipped = ((df['Rasio_Klaim'] < p1) | (df['Rasio_Klaim'] > p99)).sum()
df['Rasio_Klaim'] = df['Rasio_Klaim'].clip(lower=p1, upper=p99)
# Fill NaN rasio (dari 2 baris Biaya RS = 0 → NaN → rasio NaN) dengan median
n_nan_rasio = df['Rasio_Klaim'].isna().sum()
df['Rasio_Klaim'] = df['Rasio_Klaim'].fillna(df['Rasio_Klaim'].median())
print(f'[Isu #7] Winsorize Rasio_Klaim: P1={p1:.4f}, P99={p99:.4f}, clipped {n_clipped} baris')
print(f'         Fill NaN Rasio (dari Biaya RS=0): {n_nan_rasio} baris → median={df["Rasio_Klaim"].median():.4f}')

# 5. Selisih Nominal
df['Selisih_Klaim'] = df['Nominal Klaim Yang Disetujui'] - df['Nominal Biaya RS Yang Terjadi'].fillna(0)

# 6. Komponen waktu klaim
df['Bulan_Klaim'] = df['Tanggal Pasien Masuk RS'].dt.month
df['Tahun_Klaim'] = df['Tanggal Pasien Masuk RS'].dt.year

# 7. Kategori ICD (huruf pertama)
df['ICD_Kategori'] = df['ICD Diagnosis'].astype(str).str[0]

# 8. Binary flags (menggunakan 4 kategori lengkap)
df['is_inpatient'] = (df['Inpatient/Outpatient'].str.strip() == 'IP').astype(int)
df['is_cashless']  = (df['Reimburse/Cashless'].str.strip().str.upper() == 'C').astype(int)
df['is_odc_ods']   = df['Inpatient/Outpatient'].str.strip().isin(['ODC', 'ODS']).astype(int)

# 9. Lokasi flags (dari 4 kategori yang sudah dikonsolidasi)
df['is_overseas']  = (df['Lokasi RS'] != 'Indonesia').astype(int)  # Singapore, Malaysia, Overseas
df['is_singapore'] = (df['Lokasi RS'] == 'Singapore').astype(int)

# 10. Log nominal (untuk menangani skewness distribusi nominal)
df['log_nominal_disetujui'] = np.log1p(df['Nominal Klaim Yang Disetujui'])
df['log_biaya_rs']          = np.log1p(df['Nominal Biaya RS Yang Terjadi'].fillna(0))

print('\nFeature engineering claim-level selesai.\n')
print(f'Jumlah fitur baru: 14')
print(f'{"─" * 75}')
for col in ['Usia_Saat_Klaim', 'Lama_Rawat_Inap', 'Lama_Polis_Aktif', 'Rasio_Klaim']:
    s = df[col].describe()
    print(f'  {col:25s}  mean={s["mean"]:10.2f}  std={s["std"]:10.2f}  min={s["min"]:10.2f}  max={s["max"]:10.2f}')

# Ringkasan binary flags
print(f'\n  {"is_inpatient":25s}  mean={df["is_inpatient"].mean():.3f} (IP: {df["is_inpatient"].sum()})')
print(f'  {"is_cashless":25s}  mean={df["is_cashless"].mean():.3f}')
print(f'  {"is_odc_ods":25s}  mean={df["is_odc_ods"].mean():.3f} (ODC+ODS: {df["is_odc_ods"].sum()})')
print(f'  {"is_overseas":25s}  mean={df["is_overseas"].mean():.3f}')
print(f'  {"is_singapore":25s}  mean={df["is_singapore"].mean():.3f}')

## 6. Konstruksi Time Series Bulanan & Agregasi Historis

**Rasionalisasi:** Target prediksi berada pada level **agregat bulanan** (bukan per polis). Oleh karena itu, kita perlu mengagregasi data klaim menjadi time series bulanan. Tanggal referensi yang digunakan adalah `Tanggal Pasien Masuk RS` karena merefleksikan kapan kejadian medis terjadi (*incurred basis*), bukan kapan klaim dibayar (*paid basis*) — pendekatan standar dalam ilmu aktuaria.

Definisi target:
- **Claim_Frequency**: Jumlah klaim dalam satu bulan.
- **Total_Claim**: Total `Nominal Klaim Yang Disetujui` dalam satu bulan.
- **Claim_Severity**: Rata-rata nominal per klaim = Total_Claim / Claim_Frequency.

Selain target, kita juga menghitung fitur agregat bulanan dari data klaim (rata-rata usia, LOS, rasio, dsb.) yang akan memperkaya sinyal prediktif.

In [ ]:
# ── Agregasi Target Bulanan ────────────────────────────────────────
df['year_month'] = df['Tanggal Pasien Masuk RS'].dt.to_period('M')

monthly_targets = df.groupby('year_month').agg(
    Claim_Frequency  = ('Claim ID', 'count'),
    Total_Claim      = ('Nominal Klaim Yang Disetujui', 'sum'),
).reset_index()
monthly_targets['Claim_Severity'] = monthly_targets['Total_Claim'] / monthly_targets['Claim_Frequency']

# ── Fitur Agregat Bulanan dari Data Klaim (DIPERKAYA) ──────────────
monthly_claim_feats = df.groupby('year_month').agg(
    # Statistik usia
    avg_age          = ('Usia_Saat_Klaim', 'mean'),
    std_age          = ('Usia_Saat_Klaim', 'std'),
    median_age       = ('Usia_Saat_Klaim', 'median'),
    # Statistik lama rawat
    avg_los          = ('Lama_Rawat_Inap', 'mean'),
    max_los          = ('Lama_Rawat_Inap', 'max'),
    # Statistik rasio klaim (sudah winsorized)
    avg_rasio        = ('Rasio_Klaim', 'mean'),
    median_rasio     = ('Rasio_Klaim', 'median'),
    std_rasio        = ('Rasio_Klaim', 'std'),
    # Proporsi kategorikal
    pct_inpatient    = ('is_inpatient', 'mean'),
    pct_cashless     = ('is_cashless', 'mean'),
    pct_odc_ods      = ('is_odc_ods', 'mean'),
    pct_overseas     = ('is_overseas', 'mean'),
    pct_singapore    = ('is_singapore', 'mean'),
    # Diversitas & volume
    unique_icd       = ('ICD Diagnosis', 'nunique'),
    unique_polis     = ('Nomor Polis', 'nunique'),
    unique_lokasi    = ('Lokasi RS', 'nunique'),
    # Statistik nominal
    avg_nominal      = ('Nominal Klaim Yang Disetujui', 'mean'),
    median_nominal   = ('Nominal Klaim Yang Disetujui', 'median'),
    std_nominal      = ('Nominal Klaim Yang Disetujui', 'std'),
    max_nominal      = ('Nominal Klaim Yang Disetujui', 'max'),
    avg_biaya_rs     = ('Nominal Biaya RS Yang Terjadi', 'mean'),
    # Statistik tenure
    avg_tenure       = ('Lama_Polis_Aktif', 'mean'),
    # Log nominal
    avg_log_nominal  = ('log_nominal_disetujui', 'mean'),
).reset_index()

monthly = monthly_targets.merge(monthly_claim_feats, on='year_month')

# ── Konversi ke timestamp & tambahkan kolom bantu ──────────────────
monthly['date']      = monthly['year_month'].dt.to_timestamp()
monthly['month']     = monthly['date'].dt.month
monthly['year']      = monthly['date'].dt.year
monthly['month_idx'] = np.arange(len(monthly))  # trend index
monthly = monthly.sort_values('date').reset_index(drop=True)

print(f'Jumlah bulan observasi: {len(monthly)}')
print(f'Jumlah fitur agregat  : {len(monthly_claim_feats.columns) - 1}')
print(f'Periode: {monthly["year_month"].iloc[0]} — {monthly["year_month"].iloc[-1]}')
display(monthly[['year_month', 'Claim_Frequency', 'Claim_Severity', 'Total_Claim']])

In [ ]:
# ── Visualisasi Time Series Target ─────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
targets = ['Claim_Frequency', 'Claim_Severity', 'Total_Claim']
colors  = ['#2196F3', '#FF9800', '#4CAF50']

for ax, tgt, clr in zip(axes, targets, colors):
    ax.plot(monthly['date'], monthly[tgt], marker='o', color=clr, linewidth=2, markersize=6)
    ax.fill_between(monthly['date'], monthly[tgt], alpha=0.15, color=clr)
    ax.set_title(tgt, fontsize=13, fontweight='bold')
    ax.set_ylabel('Nilai')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Bulan')
plt.suptitle('Time Series Target — Agregat Bulanan', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Feature Engineering — Fitur Polis, Fitur Waktu & Same-Month-Last-Year

**Rasionalisasi:** Keunggulan utama pendekatan kita adalah memanfaatkan **fitur yang dapat dihitung untuk bulan-bulan yang akan diprediksi** (Agustus–Desember 2025), sehingga model dapat melakukan ekstrapolasi dengan baik.

### 7a. Fitur Berbasis Data Polis (Per Bulan)
Untuk setiap bulan (termasuk bulan prediksi), kita menghitung profil portofolio polis yang aktif:
- Jumlah polis aktif, rata-rata usia, proporsi gender, distribusi plan code.

### 7b. Fitur Waktu (*Temporal Features*)
- **Trend index**, **Cyclical encoding** (sin/cos), **Quarter**.

### 7c. Lag Features, Rolling Statistics & Same-Month-Last-Year
- **Lag 1-3**: Nilai target 1-3 bulan sebelumnya.
- **Rolling mean/std** (window 3 & 6): Momentum dan volatilitas.
- **Lag-12 (Same-Month-Last-Year)**: Untuk prediksi Aug-Dec 2025, fitur ini berisi nilai Aug-Dec 2024. Ini adalah **sinyal musiman terkuat** yang tersedia — menangkap pola tahunan yang tidak bisa ditangkap oleh lag jangka pendek.
- **YoY Ratio**: Rasio pertumbuhan year-over-year, dihitung dari 7 bulan overlap (Jan-Jul 2024 vs 2025). Memberikan informasi apakah level klaim secara keseluruhan naik atau turun.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 7a. FITUR POLIS PER BULAN (tersedia juga untuk bulan prediksi)
# ═══════════════════════════════════════════════════════════════════

# Buat daftar semua bulan: training + prediksi
all_months = pd.period_range(monthly['year_month'].min(), '2025-12', freq='M')

polis_features_list = []
for ym in all_months:
    month_end = ym.to_timestamp() + pd.offsets.MonthEnd(0)

    # Polis aktif = yang tanggal efektifnya sudah lewat
    active = df_polis[df_polis['Tanggal Efektif Polis'] <= month_end].copy()

    # Hitung usia setiap pemegang polis pada bulan tersebut
    active['age_at_month'] = (month_end - active['Tanggal Lahir']).dt.days / 365.25
    active['tenure_at_month'] = (month_end - active['Tanggal Efektif Polis']).dt.days / 365.25

    n_active = len(active)
    feat = {
        'year_month'     : ym,
        'n_active_polis' : n_active,
        'avg_age_polis'  : active['age_at_month'].mean() if n_active > 0 else 0,
        'std_age_polis'  : active['age_at_month'].std()  if n_active > 1 else 0,
        'pct_male'       : (active['Gender'] == 'M').mean() if n_active > 0 else 0.5,
        'avg_tenure_polis': active['tenure_at_month'].mean() if n_active > 0 else 0,
    }

    # Distribusi plan code
    if n_active > 0:
        plan_dist = active['Plan Code'].value_counts(normalize=True)
        for plan in ['M-001', 'M-002', 'M-003']:
            feat[f'pct_{plan}'] = plan_dist.get(plan, 0)
    else:
        for plan in ['M-001', 'M-002', 'M-003']:
            feat[f'pct_{plan}'] = 0

    polis_features_list.append(feat)

df_polis_monthly = pd.DataFrame(polis_features_list)
print(f'Fitur polis bulanan: {df_polis_monthly.shape}')
display(df_polis_monthly.tail())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 7b–c. GABUNG SEMUA FITUR + LAG + ROLLING + SAME-MONTH-LAST-YEAR
# ═══════════════════════════════════════════════════════════════════

target_cols = ['Claim_Frequency', 'Claim_Severity', 'Total_Claim']

# Fitur agregat klaim terpilih yang akan dijadikan lag features
claim_agg_cols = ['avg_nominal', 'pct_inpatient', 'unique_polis', 'pct_overseas', 'avg_rasio']

# Buat dataframe untuk seluruh bulan (training + prediksi)
df_all = pd.DataFrame({'year_month': all_months})
df_all['date']      = df_all['year_month'].apply(lambda x: x.to_timestamp())
df_all['month']     = df_all['date'].dt.month
df_all['year']      = df_all['date'].dt.year
df_all['month_idx'] = np.arange(len(df_all))  # trend
df_all['quarter']   = df_all['date'].dt.quarter

# Cyclical encoding bulan
df_all['month_sin'] = np.sin(2 * np.pi * df_all['month'] / 12)
df_all['month_cos'] = np.cos(2 * np.pi * df_all['month'] / 12)

# Merge fitur polis
df_all = df_all.merge(df_polis_monthly, on='year_month', how='left')

# Merge target + fitur agregat klaim (hanya untuk bulan training)
df_all = df_all.merge(
    monthly[['year_month'] + target_cols + claim_agg_cols],
    on='year_month', how='left'
)

# ── Lag Features & Rolling Statistics untuk TARGET ─────────────────
for tgt in target_cols:
    for lag in [1, 2, 3]:
        df_all[f'{tgt}_lag{lag}'] = df_all[tgt].shift(lag)
    # Rolling mean & std (window=3)
    df_all[f'{tgt}_rmean3'] = df_all[tgt].shift(1).rolling(3, min_periods=1).mean()
    df_all[f'{tgt}_rstd3']  = df_all[tgt].shift(1).rolling(3, min_periods=1).std().fillna(0)
    # Rolling mean (window=6)
    df_all[f'{tgt}_rmean6'] = df_all[tgt].shift(1).rolling(6, min_periods=1).mean()
    # Momentum (diff lag1)
    df_all[f'{tgt}_diff1']  = df_all[tgt].shift(1) - df_all[tgt].shift(2)

    # ── BARU: Same-Month-Last-Year (Lag-12) ──
    # Untuk bulan prediksi Aug-Dec 2025, ini adalah Aug-Dec 2024 (TERSEDIA!)
    df_all[f'{tgt}_lag12'] = df_all[tgt].shift(12)

    # ── BARU: Year-over-Year Ratio (rata-rata pertumbuhan tahunan) ──
    # Rasio antara nilai bulan ini vs bulan yang sama tahun lalu
    _yoy = df_all[tgt] / df_all[tgt].shift(12)
    # Hitung rata-rata YoY growth dari data yang tersedia
    avg_yoy = _yoy.dropna().median()
    df_all[f'{tgt}_yoy_ratio'] = _yoy.fillna(avg_yoy)
    print(f'  {tgt}: avg YoY ratio = {avg_yoy:.4f} (dari {_yoy.notna().sum()} bulan overlap)')

# ── Lag Features untuk FITUR AGREGAT KLAIM ─────────────────────────
for col in claim_agg_cols:
    df_all[f'{col}_lag1'] = df_all[col].shift(1)
    df_all[f'{col}_lag1'] = df_all[f'{col}_lag1'].ffill()

# Drop kolom agregat asli (tidak tersedia di bulan prediksi)
df_all.drop(columns=claim_agg_cols, inplace=True)

# Pisahkan training dan prediction
train_mask = df_all['Claim_Frequency'].notna()
pred_mask  = df_all['Claim_Frequency'].isna()

print(f'\nTotal bulan         : {len(df_all)}')
print(f'Bulan training      : {train_mask.sum()}')
print(f'Bulan prediksi      : {pred_mask.sum()}')
print(f'\nKolom fitur ({len(df_all.columns)}):')
print(df_all.columns.tolist())

## 8. Persiapan Dataset Pemodelan — Feature Selection

**Rasionalisasi:** Dengan hanya **16 sampel training**, jumlah fitur harus dikontrol ketat untuk mencegah *overfitting*. Berdasarkan analisis SHAP dari iterasi sebelumnya, kita melakukan **feature selection** dari 25 → ~20 fitur per target:

1. **Fitur Waktu** (4): `month_idx`, `quarter`, `month_sin`, `month_cos` — `month` dihapus karena redundan dengan sin/cos.
2. **Fitur Polis** (4): `n_active_polis`, `avg_age_polis`, `pct_male`, `avg_tenure_polis` — hanya fitur dengan SHAP impact tinggi.
3. **Lagged Claim Aggregates** (3): `avg_rasio_lag1`, `pct_inpatient_lag1`, `pct_overseas_lag1` — 3 fitur terpenting berdasarkan SHAP.
4. **Lag & Rolling Target** (9 per target): Lag 1-3, rolling mean/std 3 bulan, rolling mean 6 bulan, momentum, **lag-12 (same-month-last-year)**, dan **YoY ratio**.

Penambahan **lag-12** dan **YoY ratio** sangat kritis karena memberikan sinyal musiman yang langsung tersedia untuk bulan prediksi (Aug-Dec 2025 = Aug-Dec 2024). Pengurangan fitur polis dan claim aggregates yang low-importance mengurangi noise.

In [ ]:
# ── Definisi Fitur (DIKURANGI + DITAMBAH LAG-12) ───────────────────
# Strategi: kurangi fitur dari 25 → ~15 untuk menghindari overfitting
# pada dataset kecil (16 sampel). Fokus pada fitur high-SHAP-impact.

# Fitur waktu (dikurangi — hapus 'month' karena redundan dengan sin/cos)
time_features = ['month_idx', 'quarter', 'month_sin', 'month_cos']

# Fitur polis (dikurangi — hapus std_age & plan detail yang low-importance)
polis_features = ['n_active_polis', 'avg_age_polis', 'pct_male', 'avg_tenure_polis']

# Lagged claim aggregate features (top 3 berdasarkan SHAP, bukan 5)
claim_lag_features = ['avg_rasio_lag1', 'pct_inpatient_lag1', 'pct_overseas_lag1']

lag_features = {}
for tgt in target_cols:
    lag_features[tgt] = [
        f'{tgt}_lag1', f'{tgt}_lag2', f'{tgt}_lag3',
        f'{tgt}_rmean3', f'{tgt}_rstd3', f'{tgt}_rmean6', f'{tgt}_diff1',
        f'{tgt}_lag12',       # BARU: same-month-last-year
        f'{tgt}_yoy_ratio',   # BARU: year-over-year growth ratio
    ]

def get_feature_cols(target):
    """Mengembalikan daftar kolom fitur untuk target tertentu."""
    return time_features + polis_features + claim_lag_features + lag_features[target]

# ── Siapkan Training Data ──────────────────────────────────────────
min_train_idx = 3  # butuh minimal 3 lag
df_train = df_all[train_mask].iloc[min_train_idx:].copy()

print(f'Jumlah sampel training (setelah drop lag kosong): {len(df_train)}')
print(f'Periode training: {df_train["year_month"].iloc[0]} — {df_train["year_month"].iloc[-1]}')

for tgt in target_cols:
    feat_cols = get_feature_cols(tgt)
    print(f'\n{tgt}:')
    print(f'  Jumlah fitur  : {len(feat_cols)}')
    print(f'  Fitur         : {feat_cols}')
    print(f'  Missing di X  : {df_train[feat_cols].isnull().sum().sum()}')

## 9. Modeling & Cross-Validation — Optimasi MAPE

**Rasionalisasi:** Metrik evaluasi kompetisi adalah **MAPE (Mean Absolute Percentage Error)**. Ini mengubah seluruh strategi pemodelan:

### Mengapa MAPE Mengubah Segalanya
- **RMSE** meminimalkan error absolut kuadrat → model cenderung prediksi *mean-reverting* (flat).
- **MAPE** meminimalkan error persentase → model harus akurat secara *proporsional* untuk setiap target, terlepas dari skalanya.
- Implikasinya: prediksi flat (mendekati rata-rata) yang "aman" untuk RMSE justru buruk untuk MAPE jika data memiliki variabilitas.

### Strategi MAPE-Aware
1. **Log-transform target**: Training pada `log(y)` membuat RMSE di log-space ≈ RMSLE ≈ proxy MAPE. Model otomatis belajar relative errors.
2. **CatBoost native MAPE loss**: CatBoost mendukung `loss_function='MAPE'` langsung — mengoptimasi MAPE tanpa proxy.
3. **LightGBM objective exploration**: Optuna akan memilih antara `regression`, `huber`, dan `mape` objective — mana yang menghasilkan MAPE terendah.
4. **Ridge Regression (BARU)**: Dengan hanya 16 sampel, model linear yang *heavily regularized* menghindari overfitting yang dialami gradient boosting. Ridge secara alami menangkap trend tanpa mean-reverting.

### Cross-Validation
Tetap menggunakan **Time Series Split** (5 folds) untuk menghindari *data leakage* temporal. Metrik yang dilaporkan sekarang adalah **MAPE (%)**, bukan RMSE.

### Out-of-Fold (OOF) Predictions
Setiap model menghasilkan OOF predictions dari CV — ini digunakan untuk **optimasi bobot ensemble** secara data-driven (bukan arbitrary weights) menggunakan scipy.optimize.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FUNGSI EVALUASI & TRAINING — DIOPTIMASI UNTUK MAPE
# ═══════════════════════════════════════════════════════════════════

N_CV_SPLITS = 5  # Lebih banyak fold = lebih banyak OOF data untuk weight optimization

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — metrik evaluasi Kaggle."""
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_ts_cv(model_fn, X, y, n_splits=N_CV_SPLITS, use_log=False):
    """
    Time Series Cross-Validation — mengembalikan MAPE sebagai metrik utama.
    use_log: jika True, training dilakukan pada log(y) dan prediksi di-exp() kembali.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores_mape, scores_rmse = [], []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        if use_log:
            y_tr_input = np.log1p(y_tr)
            y_va_input = np.log1p(y_va)
            model, y_pred_log = model_fn(X_tr, y_tr_input, X_va, y_va_input)
            y_pred = np.expm1(y_pred_log)
            y_pred = np.maximum(y_pred, 0)
        else:
            model, y_pred = model_fn(X_tr, y_tr, X_va, y_va)
            y_pred = np.maximum(y_pred, 0)

        m = mape(y_va, y_pred)
        r = rmse(y_va, y_pred)
        scores_mape.append(m)
        scores_rmse.append(r)

    return np.mean(scores_mape), np.mean(scores_rmse)


def collect_oof(model_fn, X, y, n_splits=N_CV_SPLITS, use_log=False):
    """
    Kumpulkan Out-of-Fold predictions untuk ensemble weight optimization.
    Returns: (val_indices, oof_predictions) — aligned arrays.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    all_val_idx = []
    all_preds = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        if use_log:
            y_tr_input = np.log1p(y_tr)
            y_va_input = np.log1p(y_va)
            model, y_pred_log = model_fn(X_tr, y_tr_input, X_va, y_va_input)
            y_pred = np.expm1(y_pred_log)
            y_pred = np.maximum(y_pred, 0)
        else:
            model, y_pred = model_fn(X_tr, y_tr, X_va, y_va)
            y_pred = np.maximum(y_pred, 0)

        all_val_idx.extend(val_idx)
        if isinstance(y_pred, np.ndarray):
            all_preds.extend(y_pred.tolist())
        else:
            all_preds.extend(list(y_pred))

    return np.array(all_val_idx), np.array(all_preds)


def optimize_ensemble_weights(oof_dict, y_actual, min_weight=0.02):
    """
    Optimasi bobot ensemble menggunakan scipy.optimize untuk meminimalkan MAPE.
    oof_dict: {model_name: oof_predictions_array}
    y_actual: actual values aligned with OOF predictions
    """
    model_names = list(oof_dict.keys())
    n_models = len(model_names)
    oof_matrix = np.column_stack([oof_dict[m] for m in model_names])

    def objective(weights):
        pred = oof_matrix @ weights
        pred = np.maximum(pred, 1e-6)  # avoid division by zero
        return mape(y_actual, pred)

    # Constraints: weights sum to 1
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    # Bounds: each weight >= min_weight
    bounds = [(min_weight, 1.0)] * n_models

    # Multi-start optimization (avoid local minima)
    best_result = None
    best_score = np.inf

    for seed in range(20):
        rng = np.random.RandomState(seed)
        w0 = rng.dirichlet(np.ones(n_models))
        w0 = np.clip(w0, min_weight, 1.0)
        w0 /= w0.sum()

        try:
            result = minimize(objective, w0, method='SLSQP',
                            bounds=bounds, constraints=constraints,
                            options={'maxiter': 1000, 'ftol': 1e-10})
            if result.fun < best_score:
                best_score = result.fun
                best_result = result
        except:
            continue

    optimal_weights = best_result.x if best_result else np.ones(n_models) / n_models
    weight_dict = {m: w for m, w in zip(model_names, optimal_weights)}

    return weight_dict, best_score


# ── Model Factories ────────────────────────────────────────────────

def train_lgbm(X_tr, y_tr, X_va, y_va, params=None):
    default_params = {
        'objective': 'regression',
        'metric': 'mae',
        'verbosity': -1,
        'n_estimators': 500,
        'learning_rate': 0.05,
        'max_depth': 3,
        'num_leaves': 7,
        'min_child_samples': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 1.0,
        'reg_lambda': 1.0,
        'random_state': RANDOM_STATE,
    }
    if params:
        default_params.update(params)

    model = lgb.LGBMRegressor(**default_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
    )
    return model, model.predict(X_va)


def train_xgbm(X_tr, y_tr, X_va, y_va, params=None):
    default_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'verbosity': 0,
        'n_estimators': 500,
        'learning_rate': 0.05,
        'max_depth': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 1.0,
        'reg_lambda': 1.0,
        'random_state': RANDOM_STATE,
    }
    if params:
        default_params.update(params)

    model = xgb.XGBRegressor(**default_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )
    return model, model.predict(X_va)


def train_catboost(X_tr, y_tr, X_va, y_va, params=None):
    default_params = {
        'iterations': 500,
        'learning_rate': 0.05,
        'depth': 3,
        'l2_leaf_reg': 3.0,
        'loss_function': 'MAPE',
        'random_seed': RANDOM_STATE,
        'verbose': 0,
    }
    if params:
        default_params.update(params)

    model = CatBoostRegressor(**default_params)
    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        early_stopping_rounds=50,
        verbose=0
    )
    return model, model.predict(X_va)


def train_ridge(X_tr, y_tr, X_va, y_va, params=None):
    """Ridge Regression — ideal untuk small sample (n=16)."""
    alpha = params.get('alpha', 10.0) if params else 10.0
    model = Ridge(alpha=alpha, random_state=RANDOM_STATE)
    model.fit(X_tr, y_tr)
    return model, model.predict(X_va)


# ── Baseline Cross-Validation (MAPE + OOF Collection) ─────────────
print('=' * 70)
print(f'CROSS-VALIDATION BASELINE (TimeSeriesSplit, {N_CV_SPLITS} folds) — METRIK: MAPE')
print('=' * 70)

cv_results = {}

for tgt in target_cols:
    feat_cols = get_feature_cols(tgt)
    X = df_train[feat_cols].copy()
    y = df_train[tgt].copy()
    X = X.ffill().bfill().fillna(0)

    print(f'\n── {tgt} ({len(X)} sampel, {len(feat_cols)} fitur) ──')

    # Raw-scale training
    mape_lgb, rmse_lgb = evaluate_ts_cv(train_lgbm, X, y, use_log=False)
    print(f'  LightGBM  (raw)  — MAPE: {mape_lgb:>8.2f}%  RMSE: {rmse_lgb:>15,.2f}')

    mape_xgb, rmse_xgb = evaluate_ts_cv(train_xgbm, X, y, use_log=False)
    print(f'  XGBoost   (raw)  — MAPE: {mape_xgb:>8.2f}%  RMSE: {rmse_xgb:>15,.2f}')

    mape_cat, rmse_cat = evaluate_ts_cv(train_catboost, X, y, use_log=False)
    print(f'  CatBoost  (MAPE) — MAPE: {mape_cat:>8.2f}%  RMSE: {rmse_cat:>15,.2f}')

    # Ridge regression (raw + log)
    mape_ridge, rmse_ridge = evaluate_ts_cv(train_ridge, X, y, use_log=False)
    print(f'  Ridge     (raw)  — MAPE: {mape_ridge:>8.2f}%  RMSE: {rmse_ridge:>15,.2f}')

    mape_ridge_log, _ = evaluate_ts_cv(train_ridge, X, y, use_log=True)
    print(f'  Ridge     (log)  — MAPE: {mape_ridge_log:>8.2f}%')

    # Log-scale training (proxy untuk MAPE optimization)
    mape_lgb_log, _ = evaluate_ts_cv(train_lgbm, X, y, use_log=True)
    print(f'  LightGBM  (log)  — MAPE: {mape_lgb_log:>8.2f}%')

    mape_xgb_log, _ = evaluate_ts_cv(train_xgbm, X, y, use_log=True)
    print(f'  XGBoost   (log)  — MAPE: {mape_xgb_log:>8.2f}%')

    # Tentukan strategi terbaik per target
    all_mapes = {
        'lgbm_raw': mape_lgb, 'xgbm_raw': mape_xgb, 'catboost_mape': mape_cat,
        'ridge_raw': mape_ridge, 'ridge_log': mape_ridge_log,
        'lgbm_log': mape_lgb_log, 'xgbm_log': mape_xgb_log
    }
    best_key = min(all_mapes, key=all_mapes.get)
    print(f'  >>> Best: {best_key} (MAPE={all_mapes[best_key]:.2f}%)')

    cv_results[tgt] = all_mapes

## 10. Hyperparameter Tuning dengan Optuna — Optimasi MAPE

**Rasionalisasi:** *Bayesian optimization* (Optuna, *Akiba et al., 2019*) digunakan untuk mencari hyperparameter yang **langsung meminimalkan MAPE** — metrik evaluasi kompetisi.

### Strategi Tuning
1. **Objective function = MAPE**, bukan RMSE. Optuna langsung mengoptimasi metrik Kaggle.
2. **Log-transform sebagai hyperparameter**: Optuna memilih apakah training pada log(y) atau raw(y) lebih baik.
3. **LightGBM objective exploration**: Optuna memilih antara `regression`, `huber`, dan `mape`.
4. **CatBoost native MAPE loss**: Langsung `loss_function='MAPE'`.
5. **Ridge alpha tuning**: Regularization strength yang optimal untuk dataset kecil.

| Model | Trials | Tambahan Search Space |
|-------|--------|----------------------|
| **LightGBM** | 80 | + objective, + use_log |
| **XGBoost** | 60 | + use_log |
| **CatBoost** | 40 | Native MAPE loss |
| **Ridge** | 30 | + alpha, + use_log |

### OOF Prediction Collection
Setelah tuning, kita mengumpulkan **Out-of-Fold predictions** dari semua model termasuk ETS dan Seasonal Naive. OOF predictions ini digunakan untuk **optimasi bobot ensemble** secara data-driven dengan `scipy.optimize`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OPTUNA HYPERPARAMETER TUNING — OPTIMASI MAPE
# ═══════════════════════════════════════════════════════════════════

TRIALS_LGBM  = 80
TRIALS_XGB   = 60
TRIALS_CAT   = 40
TRIALS_RIDGE = 30

best_params_lgbm  = {}
best_params_xgb   = {}
best_params_cat   = {}
best_params_ridge = {}
best_log_mode     = {}  # Per target: apakah pakai log-transform?

for tgt in target_cols:
    print(f'\n{"=" * 65}')
    print(f'TUNING TARGET: {tgt} (OPTIMASI MAPE)')
    print(f'{"=" * 65}')

    feat_cols = get_feature_cols(tgt)
    X = df_train[feat_cols].copy().ffill().bfill().fillna(0)
    y = df_train[tgt].copy()

    # ── 1) LightGBM (80 trials) — termasuk pilihan objective & log ─
    def objective_lgbm(trial):
        use_log = trial.suggest_categorical('use_log', [True, False])
        obj = trial.suggest_categorical('objective', ['regression', 'huber', 'mape'])
        params = {
            'objective'        : obj,
            'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth'        : trial.suggest_int('max_depth', 2, 5),
            'num_leaves'       : trial.suggest_int('num_leaves', 4, 20),
            'min_child_samples': trial.suggest_int('min_child_samples', 2, 8),
            'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha'        : trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
            'reg_lambda'       : trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
            'n_estimators'     : trial.suggest_int('n_estimators', 50, 800),
        }
        score, _ = evaluate_ts_cv(
            lambda Xtr, ytr, Xva, yva: train_lgbm(Xtr, ytr, Xva, yva, params),
            X, y, use_log=use_log
        )
        return score

    study_lgbm = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE))
    study_lgbm.optimize(objective_lgbm, n_trials=TRIALS_LGBM, show_progress_bar=True)
    bp_lgbm = study_lgbm.best_params.copy()
    lgbm_use_log = bp_lgbm.pop('use_log')
    best_params_lgbm[tgt] = bp_lgbm

    # ── 2) XGBoost (60 trials) — termasuk pilihan log ─────────────
    def objective_xgb(trial):
        use_log = trial.suggest_categorical('use_log', [True, False])
        params = {
            'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth'        : trial.suggest_int('max_depth', 2, 5),
            'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
            'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha'        : trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
            'reg_lambda'       : trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
            'n_estimators'     : trial.suggest_int('n_estimators', 50, 800),
        }
        score, _ = evaluate_ts_cv(
            lambda Xtr, ytr, Xva, yva: train_xgbm(Xtr, ytr, Xva, yva, params),
            X, y, use_log=use_log
        )
        return score

    study_xgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE + 1))
    study_xgb.optimize(objective_xgb, n_trials=TRIALS_XGB, show_progress_bar=True)
    bp_xgb = study_xgb.best_params.copy()
    xgb_use_log = bp_xgb.pop('use_log')
    best_params_xgb[tgt] = bp_xgb

    # ── 3) CatBoost (40 trials) — native MAPE loss ───────────────
    def objective_cat(trial):
        params = {
            'learning_rate'      : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'depth'              : trial.suggest_int('depth', 2, 5),
            'l2_leaf_reg'        : trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
            'iterations'         : trial.suggest_int('iterations', 50, 800),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_strength'    : trial.suggest_float('random_strength', 0.0, 10.0),
        }
        score, _ = evaluate_ts_cv(
            lambda Xtr, ytr, Xva, yva: train_catboost(Xtr, ytr, Xva, yva, params),
            X, y, use_log=False
        )
        return score

    study_cat = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE + 2))
    study_cat.optimize(objective_cat, n_trials=TRIALS_CAT, show_progress_bar=True)
    best_params_cat[tgt] = study_cat.best_params

    # ── 4) Ridge (30 trials) — termasuk pilihan log ───────────────
    def objective_ridge(trial):
        use_log = trial.suggest_categorical('use_log', [True, False])
        params = {
            'alpha': trial.suggest_float('alpha', 0.01, 1000.0, log=True),
        }
        score, _ = evaluate_ts_cv(
            lambda Xtr, ytr, Xva, yva: train_ridge(Xtr, ytr, Xva, yva, params),
            X, y, use_log=use_log
        )
        return score

    study_ridge = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE + 3))
    study_ridge.optimize(objective_ridge, n_trials=TRIALS_RIDGE, show_progress_bar=True)
    bp_ridge = study_ridge.best_params.copy()
    ridge_use_log = bp_ridge.pop('use_log')
    best_params_ridge[tgt] = bp_ridge

    # Simpan log mode terbaik per target
    best_log_mode[tgt] = {
        'lgbm': lgbm_use_log,
        'xgbm': xgb_use_log,
        'catboost': False,
        'ridge': ridge_use_log,
    }

    print(f'\n  LightGBM Best MAPE : {study_lgbm.best_value:>8.2f}% (log={lgbm_use_log}, obj={best_params_lgbm[tgt].get("objective","regression")})')
    print(f'  XGBoost  Best MAPE : {study_xgb.best_value:>8.2f}% (log={xgb_use_log})')
    print(f'  CatBoost Best MAPE : {study_cat.best_value:>8.2f}% (native MAPE loss)')
    print(f'  Ridge    Best MAPE : {study_ridge.best_value:>8.2f}% (log={ridge_use_log}, alpha={best_params_ridge[tgt]["alpha"]:.2f})')

# ═══════════════════════════════════════════════════════════════════
# COLLECT OOF PREDICTIONS — SEMUA MODEL (untuk ensemble optimization)
# ═══════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('COLLECTING OUT-OF-FOLD PREDICTIONS')
print('=' * 70)

oof_predictions = {}  # {target: {model_name: oof_array}}
oof_actuals = {}      # {target: actual_array}

for tgt in target_cols:
    feat_cols = get_feature_cols(tgt)
    X = df_train[feat_cols].copy().ffill().bfill().fillna(0)
    y = df_train[tgt].copy()

    oof_preds_tgt = {}

    # ML Models — collect OOF
    val_idx_lgb, oof_lgb = collect_oof(
        lambda Xtr, ytr, Xva, yva: train_lgbm(Xtr, ytr, Xva, yva, best_params_lgbm[tgt]),
        X, y, use_log=best_log_mode[tgt]['lgbm']
    )
    oof_preds_tgt['lgbm'] = oof_lgb

    val_idx_xgb, oof_xgb = collect_oof(
        lambda Xtr, ytr, Xva, yva: train_xgbm(Xtr, ytr, Xva, yva, best_params_xgb[tgt]),
        X, y, use_log=best_log_mode[tgt]['xgbm']
    )
    oof_preds_tgt['xgbm'] = oof_xgb

    val_idx_cat, oof_cat = collect_oof(
        lambda Xtr, ytr, Xva, yva: train_catboost(Xtr, ytr, Xva, yva, best_params_cat[tgt]),
        X, y, use_log=False
    )
    oof_preds_tgt['catboost'] = oof_cat

    val_idx_ridge, oof_ridge = collect_oof(
        lambda Xtr, ytr, Xva, yva: train_ridge(Xtr, ytr, Xva, yva, best_params_ridge[tgt]),
        X, y, use_log=best_log_mode[tgt]['ridge']
    )
    oof_preds_tgt['ridge'] = oof_ridge

    # ETS OOF — fit ETS pada setiap training fold, forecast validation
    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
    oof_ets = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        ts_train = y.iloc[train_idx].values.astype(float)
        y_va = y.iloc[val_idx].values
        h = len(val_idx)
        try:
            ets = ExponentialSmoothing(ts_train, trend='add', seasonal=None).fit(optimized=True)
            pred_ets = ets.forecast(h)
            pred_ets = np.maximum(pred_ets, 0)
        except:
            pred_ets = np.full(h, ts_train[-3:].mean())
        oof_ets.extend(pred_ets.tolist())
    oof_preds_tgt['ets'] = np.array(oof_ets)

    # ETS Damped Trend OOF
    oof_ets_damped = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        ts_train = y.iloc[train_idx].values.astype(float)
        h = len(val_idx)
        try:
            ets_d = ExponentialSmoothing(ts_train, trend='add', damped_trend=True, seasonal=None).fit(optimized=True)
            pred_ets_d = ets_d.forecast(h)
            pred_ets_d = np.maximum(pred_ets_d, 0)
        except:
            pred_ets_d = np.full(h, ts_train[-3:].mean())
        oof_ets_damped.extend(pred_ets_d.tolist())
    oof_preds_tgt['ets_damped'] = np.array(oof_ets_damped)

    # Seasonal Naive OOF — gunakan nilai 12 bulan sebelumnya × YoY
    # df_train memiliki index yang aligned dengan df_all — kita perlukan mapping
    oof_naive = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Hitung YoY dari training fold
        train_full_idx = y_train_fold.index
        yoy_vals = []
        for ti in train_full_idx:
            # Cari index 12 bulan sebelumnya
            pos_in_all = df_all.index[df_all['year_month'] == df_train.loc[ti, 'year_month']].tolist()
            if pos_in_all:
                pos = pos_in_all[0]
                pos_12 = pos - 12
                if pos_12 >= 0 and pos_12 in df_all.index and pd.notna(df_all.loc[pos_12, tgt]):
                    yoy_vals.append(df_all.loc[pos, tgt] / df_all.loc[pos_12, tgt])

        median_yoy = np.median(yoy_vals) if yoy_vals else 1.0

        naive_preds_fold = []
        for vi in val_idx:
            actual_idx = y.index[vi]
            val_ym = df_train.loc[actual_idx, 'year_month']
            pos_in_all = df_all.index[df_all['year_month'] == val_ym].tolist()
            if pos_in_all:
                pos = pos_in_all[0]
                pos_12 = pos - 12
                if pos_12 >= 0 and pos_12 in df_all.index and pd.notna(df_all.loc[pos_12, tgt]):
                    naive_preds_fold.append(df_all.loc[pos_12, tgt] * median_yoy)
                else:
                    # Fallback: rata-rata 3 bulan terakhir training
                    naive_preds_fold.append(y_train_fold.iloc[-3:].mean())
            else:
                naive_preds_fold.append(y_train_fold.iloc[-3:].mean())
        oof_naive.extend(naive_preds_fold)
    oof_preds_tgt['naive'] = np.array(oof_naive)

    # Simpan OOF
    oof_actuals_arr = y.iloc[val_idx_lgb].values  # same indices for all models
    oof_predictions[tgt] = oof_preds_tgt
    oof_actuals[tgt] = oof_actuals_arr

    # Print individual model MAPE
    print(f'\n  {tgt} — OOF MAPE per model:')
    for m_name, m_oof in oof_preds_tgt.items():
        m_mape = mape(oof_actuals_arr, m_oof)
        print(f'    {m_name:12s}: {m_mape:>8.2f}%')

# ═══════════════════════════════════════════════════════════════════
# OPTIMASI BOBOT ENSEMBLE — DATA-DRIVEN (scipy.optimize)
# ═══════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('OPTIMASI BOBOT ENSEMBLE (scipy.optimize MAPE)')
print('=' * 70)

optimal_weights = {}  # {target: {model_name: weight}}

for tgt in target_cols:
    weights, best_mape = optimize_ensemble_weights(
        oof_predictions[tgt], oof_actuals[tgt], min_weight=0.02
    )
    optimal_weights[tgt] = weights

    print(f'\n  {tgt} — Optimal weights (OOF MAPE = {best_mape:.2f}%):')
    for m, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f'    {m:12s}: {w:.4f} ({w*100:.1f}%)')

    # Sanity check: juga coba equal weighting
    equal_oof = np.mean([oof_predictions[tgt][m] for m in oof_predictions[tgt]], axis=0)
    equal_mape = mape(oof_actuals[tgt], equal_oof)
    print(f'    {"(equal avg)":12s}: {equal_mape:.2f}%')

## 11. Pelatihan Model Final & Prediksi Recursive Multi-Step

**Rasionalisasi:** Setelah mendapatkan hyperparameter optimal, **mode training terbaik** (log vs raw), dan **bobot ensemble optimal** dari OOF, kita melatih model akhir menggunakan seluruh data training.

### 7 Komponen Prediksi
1. **LightGBM** (tuned, log/raw sesuai Optuna) — Objective terbaik dipilih dari {regression, huber, mape}.
2. **XGBoost** (tuned, log/raw sesuai Optuna).
3. **CatBoost** (tuned, native MAPE loss) — Langsung mengoptimasi persentase error.
4. **Ridge Regression** (tuned, log/raw sesuai Optuna) — Model linear *heavily regularized*, ideal untuk small samples (n=16).
5. **Exponential Smoothing** (Holt's Linear Trend) — Baseline statistik.
6. **ETS Damped Trend** (BARU) — Varian ETS dengan *damped trend* yang mencegah ekstrapolasi berlebihan.
7. **Seasonal Naive** — Nilai bulan yang sama tahun lalu × YoY growth ratio. Model sederhana namun powerful karena langsung menangkap pola musiman.

### Recursive Forecasting
Untuk lag features, kita menggunakan **weighted ensemble** (bobot optimal) sebagai nilai "aktual" untuk prediksi bulan berikutnya.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TRAIN FINAL MODELS & RECURSIVE PREDICTION (MAPE-AWARE)
# ═══════════════════════════════════════════════════════════════════

final_predictions = {}  # {target: {model_name: [pred_m1, ..., pred_m5]}}
trained_models    = {}  # Simpan model untuk SHAP nanti

for tgt in target_cols:
    print(f'\n{"+" * 65}')
    print(f'TARGET: {tgt}')
    print(f'{"+" * 65}')

    feat_cols = get_feature_cols(tgt)
    X_train_full = df_train[feat_cols].copy().ffill().bfill().fillna(0)
    y_train_full = df_train[tgt].copy()

    # ── 1) LightGBM (tuned) ──
    lgbm_params = best_params_lgbm[tgt].copy()
    lgbm_use_log = best_log_mode[tgt]['lgbm']
    y_lgbm = np.log1p(y_train_full) if lgbm_use_log else y_train_full
    lgbm_final = lgb.LGBMRegressor(
        metric='mae', verbosity=-1,
        random_state=RANDOM_STATE, **lgbm_params
    )
    lgbm_final.fit(X_train_full, y_lgbm)
    trained_models[f'{tgt}_lgbm'] = lgbm_final
    print(f'  LightGBM (tuned, log={lgbm_use_log}) trained.')

    # ── 2) XGBoost (tuned) ──
    xgb_params = best_params_xgb[tgt].copy()
    xgb_use_log = best_log_mode[tgt]['xgbm']
    y_xgb = np.log1p(y_train_full) if xgb_use_log else y_train_full
    xgb_final = xgb.XGBRegressor(
        objective='reg:squarederror', eval_metric='mae', verbosity=0,
        random_state=RANDOM_STATE, **xgb_params
    )
    xgb_final.fit(X_train_full, y_xgb)
    trained_models[f'{tgt}_xgbm'] = xgb_final
    print(f'  XGBoost (tuned, log={xgb_use_log}) trained.')

    # ── 3) CatBoost (tuned, native MAPE loss) ──
    cat_params = best_params_cat[tgt].copy()
    cat_final = CatBoostRegressor(
        loss_function='MAPE',
        random_seed=RANDOM_STATE, verbose=0, **cat_params
    )
    cat_final.fit(X_train_full, y_train_full)
    trained_models[f'{tgt}_cat'] = cat_final
    print(f'  CatBoost (tuned, MAPE loss) trained.')

    # ── 4) Ridge Regression (tuned) ──
    ridge_params = best_params_ridge[tgt].copy()
    ridge_use_log = best_log_mode[tgt]['ridge']
    y_ridge = np.log1p(y_train_full) if ridge_use_log else y_train_full
    ridge_final = Ridge(alpha=ridge_params['alpha'], random_state=RANDOM_STATE)
    ridge_final.fit(X_train_full, y_ridge)
    trained_models[f'{tgt}_ridge'] = ridge_final
    print(f'  Ridge (tuned, log={ridge_use_log}, alpha={ridge_params["alpha"]:.2f}) trained.')

    # ── 5) Exponential Smoothing (Holt's Linear) ──
    ts_series = monthly[tgt].values.astype(float)
    try:
        ets_model = ExponentialSmoothing(
            ts_series, trend='add', seasonal=None
        ).fit(optimized=True)
        ets_pred = ets_model.forecast(5)
        ets_pred = np.maximum(ets_pred, 0)
        print(f'  ETS (linear trend) trained.')
    except Exception as e:
        print(f'  ETS gagal: {e}. Fallback = mean 3 bulan terakhir.')
        ets_pred = np.full(5, ts_series[-3:].mean())

    # ── 6) ETS Damped Trend ──
    try:
        ets_damped_model = ExponentialSmoothing(
            ts_series, trend='add', damped_trend=True, seasonal=None
        ).fit(optimized=True)
        ets_damped_pred = ets_damped_model.forecast(5)
        ets_damped_pred = np.maximum(ets_damped_pred, 0)
        print(f'  ETS (damped trend) trained.')
    except Exception as e:
        print(f'  ETS damped gagal: {e}. Fallback = ETS linear.')
        ets_damped_pred = ets_pred.copy()

    # ── 7) Seasonal Naive (Same-Month-Last-Year + YoY Adjustment) ──
    monthly_vals = monthly.set_index('year_month')[tgt]
    naive_preds = []
    yoy_ratios = []
    for m_offset in range(7):  # Jan-Jul = 7 bulan overlap
        ym_2024 = pd.Period(f'2024-{m_offset+1:02d}', freq='M')
        ym_2025 = pd.Period(f'2025-{m_offset+1:02d}', freq='M')
        if ym_2024 in monthly_vals.index and ym_2025 in monthly_vals.index:
            ratio = monthly_vals[ym_2025] / monthly_vals[ym_2024]
            yoy_ratios.append(ratio)
    avg_yoy_growth = np.median(yoy_ratios) if yoy_ratios else 1.0
    print(f'  Seasonal Naive: YoY growth = {avg_yoy_growth:.4f} (dari {len(yoy_ratios)} bulan)')

    for pred_month in [8, 9, 10, 11, 12]:
        ym_last_year = pd.Period(f'2024-{pred_month:02d}', freq='M')
        if ym_last_year in monthly_vals.index:
            naive_preds.append(monthly_vals[ym_last_year] * avg_yoy_growth)
        else:
            naive_preds.append(ts_series[-3:].mean())
    naive_pred = np.array(naive_preds)
    naive_pred = np.maximum(naive_pred, 0)
    print(f'  Seasonal Naive: {[f"{v:,.0f}" for v in naive_pred]}')

    # ── Recursive Forecasting untuk ML Models ──
    df_pred = df_all.copy()
    pred_indices = df_pred[pred_mask].index.tolist()

    ml_preds = {'lgbm': [], 'xgbm': [], 'catboost': [], 'ridge': []}
    models_dict = {
        'lgbm': (lgbm_final, lgbm_use_log),
        'xgbm': (xgb_final, xgb_use_log),
        'catboost': (cat_final, False),
        'ridge': (ridge_final, ridge_use_log),
    }

    # Bobot optimal dari OOF (hanya ML models untuk weighted lag update)
    tgt_weights = optimal_weights[tgt]
    ml_model_names = ['lgbm', 'xgbm', 'catboost', 'ridge']
    ml_weight_sum = sum(tgt_weights.get(m, 0) for m in ml_model_names)

    for idx in pred_indices:
        # Update lag features
        for lag in [1, 2, 3]:
            prev_idx = idx - lag
            if prev_idx >= 0:
                df_pred.loc[idx, f'{tgt}_lag{lag}'] = df_pred.loc[prev_idx, tgt]

        # Update rolling features
        recent_vals = []
        for back in range(1, 7):
            if idx - back >= 0 and pd.notna(df_pred.loc[idx - back, tgt]):
                recent_vals.append(df_pred.loc[idx - back, tgt])

        if len(recent_vals) >= 1:
            df_pred.loc[idx, f'{tgt}_rmean3'] = np.mean(recent_vals[:3])
            df_pred.loc[idx, f'{tgt}_rstd3']  = np.std(recent_vals[:3]) if len(recent_vals[:3]) > 1 else 0
            df_pred.loc[idx, f'{tgt}_rmean6'] = np.mean(recent_vals[:6])

        if len(recent_vals) >= 2:
            df_pred.loc[idx, f'{tgt}_diff1'] = recent_vals[0] - recent_vals[1]

        # Prediksi
        X_row = df_pred.loc[[idx], feat_cols].ffill().bfill().fillna(0)

        for model_name, (model, use_log) in models_dict.items():
            pred_val = model.predict(X_row)[0]
            if use_log:
                pred_val = np.expm1(pred_val)
            pred_val = max(pred_val, 0)
            ml_preds[model_name].append(pred_val)

        # Gunakan weighted ensemble (bobot optimal) untuk lag berikutnya
        step_idx = len(ml_preds['lgbm']) - 1  # 0-indexed step
        weighted_pred = 0
        for m_name in ml_model_names:
            w = tgt_weights.get(m_name, 0) / ml_weight_sum if ml_weight_sum > 0 else 0.25
            weighted_pred += w * ml_preds[m_name][-1]

        # Blend dengan seasonal naive untuk lag update (prevents flat predictions)
        if step_idx < len(naive_pred):
            blend_ratio = 0.6  # 60% ML weighted, 40% seasonal naive
            lag_update_val = blend_ratio * weighted_pred + (1 - blend_ratio) * naive_pred[step_idx]
        else:
            lag_update_val = weighted_pred

        df_pred.loc[idx, tgt] = lag_update_val

    # Simpan prediksi
    final_predictions[tgt] = {
        'lgbm': np.array(ml_preds['lgbm']),
        'xgbm': np.array(ml_preds['xgbm']),
        'catboost': np.array(ml_preds['catboost']),
        'ridge': np.array(ml_preds['ridge']),
        'ets': ets_pred,
        'ets_damped': ets_damped_pred,
        'naive': naive_pred,
    }

    print(f'\n  Prediksi per model:')
    for m_name in final_predictions[tgt]:
        vals = final_predictions[tgt][m_name]
        print(f'    {m_name:12s}: {[f"{v:,.0f}" for v in vals]}')

## 12. Ensemble — Data-Driven Optimal Weighting

**Rasionalisasi:** Ensemble mengurangi *variance* prediksi (*Breiman, 1996*). Perbedaan kunci dari pendekatan sebelumnya:

### Bobot Optimal dari OOF (bukan arbitrary)
- **Sebelumnya**: Bobot tetap 70% ML + 15% ETS + 15% Naive (arbitrary).
- **Sekarang**: Bobot dihitung melalui **scipy.optimize** yang meminimalkan MAPE pada **Out-of-Fold predictions**. Optimizer mencoba 20 starting points berbeda (multi-start SLSQP) untuk menghindari local minima.
- Setiap model mendapat bobot minimum 2% untuk menjaga diversitas.

### Consistency Adjustment (lebih ringan)
- Total_Claim = 70% prediksi langsung + 30% (Frequency × Severity).
- Severity di-update agar konsisten: Severity = Total / Frequency.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ENSEMBLE — DATA-DRIVEN OPTIMAL WEIGHTING
# ═══════════════════════════════════════════════════════════════════

ensemble_preds = {}

for tgt in target_cols:
    preds = final_predictions[tgt]
    weights = optimal_weights[tgt]

    # Pastikan semua model ada di weights (fallback jika ada yang missing)
    all_models = list(preds.keys())
    for m in all_models:
        if m not in weights:
            weights[m] = 0.02  # minimum weight

    # Normalisasi bobot agar sum = 1 (hanya untuk model yang ada di preds)
    total_w = sum(weights.get(m, 0) for m in all_models)
    norm_weights = {m: weights.get(m, 0) / total_w for m in all_models}

    print(f'{tgt} -- Bobot ensemble (OOF-optimized):')
    for m in sorted(norm_weights.keys(), key=lambda x: -norm_weights[x]):
        print(f'  {m:12s}: {norm_weights[m]:.4f} ({norm_weights[m]*100:.1f}%)')

    # Weighted average
    ensemble = sum(norm_weights[m] * preds[m] for m in all_models)
    ensemble = np.maximum(ensemble, 0)
    ensemble_preds[tgt] = ensemble
    print(f'  Prediksi : {[f"{v:,.0f}" for v in ensemble]}')
    print()

# ── Consistency Adjustment (70:30 direct vs product) ───────────────
total_from_product = ensemble_preds['Claim_Frequency'] * ensemble_preds['Claim_Severity']
total_direct       = ensemble_preds['Total_Claim']

ensemble_preds['Total_Claim'] = 0.7 * total_direct + 0.3 * total_from_product
ensemble_preds['Claim_Severity'] = ensemble_preds['Total_Claim'] / np.maximum(ensemble_preds['Claim_Frequency'], 1)

print('\n=== PREDIKSI FINAL (setelah consistency adjustment) ===')
pred_months = [8, 9, 10, 11, 12]
for i, m in enumerate(pred_months):
    print(f'  2025-{m:02d}: Freq={ensemble_preds["Claim_Frequency"][i]:.1f}, '
          f'Severity={ensemble_preds["Claim_Severity"][i]:,.0f}, '
          f'Total={ensemble_preds["Total_Claim"][i]:,.0f}')

# ── Perbandingan dengan pendekatan sebelumnya ──────────────────────
print('\n=== PERBANDINGAN STRATEGI ENSEMBLE ===')
for tgt in target_cols:
    preds = final_predictions[tgt]

    # Equal weight
    equal_pred = np.mean([preds[m] for m in preds], axis=0)

    # Pure seasonal naive
    naive_only = preds['naive']

    print(f'\n  {tgt}:')
    print(f'    Optimal Ensemble : {[f"{v:,.0f}" for v in ensemble_preds[tgt]]}')
    print(f'    Equal Average    : {[f"{v:,.0f}" for v in equal_pred]}')
    print(f'    Seasonal Naive   : {[f"{v:,.0f}" for v in naive_only]}')

## 13. Analisis Feature Importance & SHAP Values

**Rasionalisasi:** Interpretabilitas model merupakan aspek kritis dalam penelitian akademis dan pengambilan keputusan bisnis asuransi. Kita menggunakan dua pendekatan:

1. **Tree-based Feature Importance** (Split/Gain): Mengukur berapa kali fitur digunakan untuk splitting dan seberapa besar kontribusinya terhadap penurunan loss.
2. **SHAP (SHapley Additive exPlanations, Lundberg & Lee, 2017)**: Berbasis teori permainan (*cooperative game theory*), memberikan kontribusi marjinal setiap fitur terhadap prediksi individual. SHAP memiliki keunggulan:
   - *Local interpretability*: Menjelaskan setiap prediksi.
   - *Global interpretability*: Agregasi untuk melihat pola umum.
   - *Consistency*: Fitur yang berkontribusi lebih besar selalu mendapat nilai SHAP lebih tinggi.

Analisis ini membantu menjawab pertanyaan: **Faktor apa yang paling memengaruhi frekuensi, severitas, dan total klaim asuransi kesehatan?**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FEATURE IMPORTANCE & SHAP ANALYSIS
# (Menggunakan LightGBM tuned — model tree terbaik untuk interpretasi)
# ═══════════════════════════════════════════════════════════════════

# ── Tree-based Feature Importance ──────────────────────────────────
fig_fi, axes_fi = plt.subplots(1, 3, figsize=(20, 7))

for i, tgt in enumerate(target_cols):
    model_key = f'{tgt}_lgbm'
    model = trained_models[model_key]
    feat_cols = get_feature_cols(tgt)

    importance = model.feature_importances_
    feat_imp = pd.DataFrame({'feature': feat_cols, 'importance': importance})
    feat_imp = feat_imp.sort_values('importance', ascending=True).tail(15)

    axes_fi[i].barh(feat_imp['feature'], feat_imp['importance'], color=colors[i])
    axes_fi[i].set_title(f'Feature Importance\n{tgt}', fontweight='bold')
    axes_fi[i].set_xlabel('Importance (Split)')

plt.suptitle('Tree-based Feature Importance (LightGBM Tuned)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── SHAP Values ────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('SHAP VALUES ANALYSIS')
print('=' * 65)

for tgt in target_cols:
    model_key = f'{tgt}_lgbm'
    model = trained_models[model_key]
    feat_cols = get_feature_cols(tgt)

    X_shap = df_train[feat_cols].copy().ffill().bfill().fillna(0)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)

    print(f'\n-- SHAP Summary: {tgt} --')
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=15)
    plt.title(f'SHAP Feature Importance - {tgt}', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Beeswarm plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_shap, show=False, max_display=15)
    plt.title(f'SHAP Beeswarm - {tgt}', fontweight='bold')
    plt.tight_layout()
    plt.show()

### Interpretasi SHAP Values (Narasi Akademis)

Berdasarkan analisis SHAP, berikut temuan utama yang relevan untuk jurnal metodologi:

1. **Fitur Lag Target (lag1, lag2, rolling mean)** — Konsisten menjadi prediktor terpenting di ketiga target. Hal ini menunjukkan bahwa klaim asuransi kesehatan memiliki **dependensi temporal kuat** (*autocorrelation*): pola klaim bulan lalu sangat informatif untuk memprediksi bulan depan. Temuan ini sejalan dengan teori aktuaria bahwa *claim development* mengikuti pola yang relatif stabil dalam jangka pendek.

2. **Lagged Claim Aggregates (avg_nominal_lag1, pct_inpatient_lag1, unique_polis_lag1, pct_overseas_lag1, avg_rasio_lag1)** — Fitur-fitur ini memberikan sinyal tentang **komposisi klaim bulan sebelumnya**: proporsi rawat inap, rata-rata nominal klaim, jumlah polis aktif yang klaim, proporsi klaim overseas, dan rata-rata rasio persetujuan. Perubahan komposisi ini bersifat prediktif karena mencerminkan pergeseran profil risiko portofolio.

3. **Trend Index (month_idx)** — Menunjukkan adanya **tren sekuler** dalam portofolio asuransi, kemungkinan disebabkan oleh pertumbuhan jumlah polis aktif dan/atau inflasi biaya medis.

4. **Fitur Polis (n_active_polis, avg_age_polis)** — Jumlah polis aktif berpengaruh langsung terhadap frekuensi klaim (*exposure effect*). Rata-rata usia portofolio memengaruhi severity karena populasi yang lebih tua cenderung memiliki klaim dengan biaya lebih tinggi.

5. **Cyclical Features (month_sin, month_cos)** — Menangkap pola musiman (*seasonality*), seperti peningkatan klaim pada musim hujan (penyakit infeksi) atau awal tahun (pemanfaatan benefit tahunan).

6. **Distribusi Plan Code** — Plan M-001 (Worldwide) dan M-002 (Asia) cenderung memiliki severity lebih tinggi karena mencakup perawatan di luar negeri dengan biaya lebih mahal.

## 14. Generasi File Submission

**Rasionalisasi:** File submission harus memiliki format yang **identik** dengan `sample_submission.csv`. Kita memetakan prediksi ensemble ke format `id` yang sesuai (YYYY_MM_MetricName) dan menyimpannya sebagai `final_submission.csv`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GENERASI FILE SUBMISSION
# ═══════════════════════════════════════════════════════════════════

submission = pd.read_csv('sample_submission.csv')

pred_months = [8, 9, 10, 11, 12]
metric_map = {
    'Claim_Frequency': ensemble_preds['Claim_Frequency'],
    'Claim_Severity' : ensemble_preds['Claim_Severity'],
    'Total_Claim'    : ensemble_preds['Total_Claim'],
}

for idx, row in submission.iterrows():
    parts  = row['id'].split('_')
    year   = int(parts[0])
    month  = int(parts[1])
    metric = '_'.join(parts[2:])

    month_idx = pred_months.index(month)
    submission.loc[idx, 'value'] = metric_map[metric][month_idx]

# Pastikan Claim_Frequency adalah integer (jumlah klaim)
for idx, row in submission.iterrows():
    metric = '_'.join(row['id'].split('_')[2:])
    if metric == 'Claim_Frequency':
        submission.loc[idx, 'value'] = round(submission.loc[idx, 'value'])

print('=== FILE SUBMISSION ===')
display(submission)

submission.to_csv('final_submission.csv', index=False)
print('\nFile disimpan sebagai: final_submission.csv')

In [ ]:
# ── Verifikasi: Bandingkan format dengan sample_submission ─────────
sample = pd.read_csv('sample_submission.csv')
result = pd.read_csv('final_submission.csv')

print('Verifikasi format submission:')
print(f'  Kolom sample  : {sample.columns.tolist()}')
print(f'  Kolom result  : {result.columns.tolist()}')
print(f'  Jumlah baris  : sample={len(sample)}, result={len(result)}')
print(f'  ID match      : {(sample["id"] == result["id"]).all()}')
print(f'  Ada NaN?      : {result["value"].isna().any()}')
print(f'  Ada negatif?  : {(result["value"] < 0).any()}')
print(f'\nSemua terverifikasi. Submission siap di-upload ke Kaggle!')

## 15. Visualisasi Prediksi vs Historis

**Rasionalisasi:** Visualisasi perbandingan antara data historis dan prediksi memberikan validasi kualitatif (*sanity check*) terhadap kewajaran hasil model. Prediksi yang baik seharusnya menunjukkan kesinambungan tren dan level yang masuk akal relatif terhadap data historis.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VISUALISASI PREDIKSI vs HISTORIS
# ═══════════════════════════════════════════════════════════════════

pred_dates = pd.date_range('2025-08-01', periods=5, freq='MS')

fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=False)

for ax, tgt, clr in zip(axes, target_cols, colors):
    # Historis
    ax.plot(monthly['date'], monthly[tgt], 'o-', color=clr, linewidth=2,
            markersize=6, label='Historis (Aktual)', zorder=3)
    # Prediksi
    ax.plot(pred_dates, ensemble_preds[tgt], 's--', color='red', linewidth=2,
            markersize=8, label='Prediksi (Ensemble)', zorder=4)

    # Shaded area untuk prediksi
    ax.axvspan(pred_dates[0], pred_dates[-1], alpha=0.08, color='red', label='Forecast Horizon')

    ax.set_title(tgt, fontsize=13, fontweight='bold')
    ax.set_ylabel('Nilai')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Bulan')
plt.suptitle('Prediksi Klaim Asuransi -- Agustus s.d. Desember 2025',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 16. Kesimpulan & Ringkasan Metodologi

### Ringkasan Pipeline

| Tahap | Metode | Alasan |
|-------|--------|--------|
| **Data Audit** | 8-point quality audit | Mengidentifikasi 7 isu kritis sebelum cleaning |
| **Cleaning** | Imputasi IP/OP (LOS), median-delay, ICD Z99, Lokasi RS (plan), konsolidasi 11→4, winsorize rasio | **0 baris di-drop**; memaksimalkan akurasi agregat bulanan |
| **Feature Engineering** | Usia, LOS, tenure, rasio, binary flags, lag-12 (same-month-last-year), YoY ratio | Domain-driven + tersedia di horizon prediksi; ~20 fitur/target |
| **Modeling** | LightGBM + XGBoost + CatBoost + **Ridge** + ETS + ETS Damped + **Seasonal Naive** | **7 model** dari 3 paradigma: gradient boosting, linear, statistik |
| **Tuning** | Optuna TPE — LGB:80 + XGB:60 + CAT:40 + **Ridge:30** trials/target | Optimasi langsung terhadap **MAPE** (metrik Kaggle); termasuk log-transform sebagai hyperparameter |
| **Validation** | TimeSeriesSplit (**5 folds**) + OOF prediction collection | Lebih banyak fold = lebih banyak data untuk weight optimization |
| **Ensemble** | **scipy.optimize** (20× multi-start SLSQP) pada OOF predictions | Bobot optimal data-driven, bukan arbitrary — langsung meminimalkan MAPE |
| **Interpretasi** | SHAP values + tree importance | Interpretabilitas model untuk analisis bisnis dan akademis |

### Kontribusi Utama
1. **Comprehensive data cleaning** dengan 0 data loss — 7 isu ditangani melalui imputasi cerdas berbasis domain.
2. **MAPE-first optimization**: Seluruh pipeline (loss function, Optuna objective, ensemble weighting) dioptimasi untuk MAPE — metrik evaluasi Kaggle.
3. **7-model ensemble** dari 3 paradigma berbeda: gradient boosting (LightGBM, XGBoost, CatBoost), model linear (Ridge), dan metode statistik (ETS, ETS Damped, Seasonal Naive). Diversitas ini mengurangi risiko overfitting.
4. **OOF-optimized ensemble weights**: Bobot dihitung secara data-driven menggunakan scipy.optimize pada Out-of-Fold predictions — bukan arbitrary splits (70/15/15).
5. **Seasonal Naive dengan YoY adjustment**: Menangkap pola musiman yang tidak bisa ditangkap gradient boosting pada 16 sampel.
6. **Ridge Regression**: Model linear heavily-regularized yang ideal untuk small sample (n=16), menangkap trend tanpa overfitting.
7. **Log-transform sebagai hyperparameter**: Optuna memilih apakah training pada log(y) atau raw(y) lebih baik — proxy untuk optimasi MAPE.
8. **Lag-12 (Same-Month-Last-Year)** dan **YoY ratio**: Fitur musiman yang tersedia untuk bulan prediksi, memberikan sinyal terkuat untuk forecasting.

### Referensi
- Akiba, T. et al. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework*. KDD.
- Lundberg, S. & Lee, S. (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS.
- Ke, G. et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*. NeurIPS.
- Chen, T. & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD.
- Prokhorenkova, L. et al. (2018). *CatBoost: Unbiased Boosting with Categorical Features*. NeurIPS.
- Hoerl, A.E. & Kennard, R.W. (1970). *Ridge Regression: Biased Estimation for Nonorthogonal Problems*. Technometrics.
- Breiman, L. (1996). *Bagging Predictors*. Machine Learning.
- Hyndman, R.J. & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*, 3rd ed.
- Tukey, J.W. (1977). *Exploratory Data Analysis*. Addison-Wesley.

print('Pipeline selesai. Notebook siap dijalankan.')